# Configurable PII guardrails by geography
**Practical notebook 01 · Unvibecode**

A customer sends: “Refund my $10 order. Email alex@example.com.” The refund request can reach the assistant without the email address. This notebook detects selected personal identifiers and either replaces their spans or blocks the request before a downstream call.

Set the countries your application handles, then configure entity thresholds and actions. The detector is **Presidio**, with a local spaCy model for names and locations. It works independently of your application’s LLM; no API key or GPU is required.

**Contents:** Setup · Geography and policy · Implementation · Synthetic samples · Verification · Latency · Integration

Geography selects identifier recognizers; it does not determine legal compliance or language support. This notebook handles **English text**. An `ALLOW` result means no configured match was found, not that the text contains no personal data.

## 1. Run the notebook
Tested with Python 3.12. Use a fresh environment to avoid changing dependencies in an existing application.

```bash
python -m venv .venv
```
Activate it with `source .venv/bin/activate` on macOS/Linux, or `.venv\Scripts\Activate.ps1` in Windows PowerShell. Then:

```bash
python -m pip install jupyterlab
python -m jupyterlab
```

Open this notebook. On the first run, set `INSTALL_DEPENDENCIES = True` below and run **only that cell**. Restart the kernel, set it back to `False`, and choose **Run All**. Installation downloads packages and the English model. Scanning uses local resources.

All included data is synthetic test data, not sourced personal records. Structurally valid identifiers are not guaranteed to be unassigned; do not use these fixtures to contact people or authenticate against live services.

In [ ]:
import subprocess
import sys

INSTALL_DEPENDENCIES = False
PACKAGES = [
    "presidio-analyzer==2.2.364",
    "presidio-anonymizer==2.2.364",
    "spacy==3.8.16",
    "tldextract==5.3.2",
    "phonenumbers==9.0.40",
    "regex==2026.9.10",
]
MODEL_URL = (
    "https://github.com/explosion/spacy-models/releases/download/"
    "en_core_web_sm-3.8.0/en_core_web_sm-3.8.0-py3-none-any.whl"
)
if INSTALL_DEPENDENCIES:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *PACKAGES, MODEL_URL])
    print("Installed. Restart the kernel, set INSTALL_DEPENDENCIES=False, then Run All.")
else:
    print("Installation skipped. Use the environment prepared above.")

## 2. Geography selects coverage
Use one country or a union of countries. Select the profile from trusted application configuration and known data origins. A caller-supplied country must not be allowed to weaken a mandatory deployment profile. Do not infer data origin solely from IP address, residence, or UI language.

| Input | Additional identifiers enabled |
|---|---|
| `IN` / `India` | PAN, Aadhaar, Indian passport |
| `US` / `USA` | SSN, ITIN, US bank account |
| `GB` / `UK` | National Insurance number, NHS number |
| `SG` / `Singapore` | NRIC / FIN |
| `AU` / `Australia` | Tax File Number, Medicare number |
| `CA` / `Canada` | Social Insurance Number |

Every profile includes email, phone, credit card, IBAN, and IP checks. Names are enabled by default; locations are optional. These are selected recognizers, **not exhaustive national coverage**. Postal addresses, dates of birth, account usernames, free-form health information, and many national identifiers need additional rules or models. Phone parsing uses the selected regions; explicit international numbers may also be recognized.

Unknown geographies and unsupported declared languages are rejected. For mixed-country records, use a union such as `["IN", "US"]`. Adding countries can increase false positives because identifiers overlap. Language is a separate setting; this implementation does not automatically identify the actual language of a string.

## 3. Detection and release policy
The implementation has four parts: validated configuration, explicit recognizer loading, a scan result, and a release gate.

- `redact` replaces detected spans with entity labels; `block` releases no text.
- Per-entity thresholds override the default. Scores are recognizer scores, not calibrated probabilities. `0.35` is a demonstration starting point, not a recommended production optimum.
- Overlapping detections are merged before replacement. Conflicting entity labels become `<PII>` so the entire detected span is removed.
- Oversized input, invalid detector output, or a scanner exception blocks release. Missing dependencies prevent initialization.
- Audit records contain decisions, counts, configuration ID, and timing. They exclude input text and detected values. Redacted text can still contain missed PII; never treat it as safe diagnostic data.

Email validation uses the public-suffix snapshot bundled with `tldextract` instead of fetching a list during a request. Country recognizers are instantiated explicitly because some are disabled in Presidio’s default registry.

In [ ]:
from collections import Counter
from dataclasses import dataclass, field, replace
from functools import lru_cache
from importlib.metadata import version
from math import isfinite
from time import perf_counter
from types import MappingProxyType
from typing import Mapping
import hashlib
import json

import spacy
import tldextract
from presidio_analyzer import AnalyzerEngine, RecognizerRegistry, RecognizerResult
from presidio_analyzer.nlp_engine import NlpEngineProvider
from presidio_analyzer import predefined_recognizers as recognizers
from presidio_anonymizer import AnonymizerEngine
from presidio_anonymizer.entities import OperatorConfig


PROFILE_VERSION = "1.0.0"
MODEL_NAME = "en_core_web_sm"
GEO_ENTITIES = {
    "IN": ("IN_PAN", "IN_AADHAAR", "IN_PASSPORT"),
    "US": ("US_SSN", "US_ITIN", "US_BANK_NUMBER"),
    "GB": ("UK_NINO", "UK_NHS"),
    "SG": ("SG_NRIC_FIN",),
    "AU": ("AU_TFN", "AU_MEDICARE"),
    "CA": ("CA_SIN",),
}
COMMON_ENTITIES = (
    "EMAIL_ADDRESS", "PHONE_NUMBER", "CREDIT_CARD", "IBAN_CODE", "IP_ADDRESS",
)
ALIASES = {
    "INDIA": "IN", "USA": "US", "UNITED STATES": "US",
    "UK": "GB", "UNITED KINGDOM": "GB", "SINGAPORE": "SG",
    "AUSTRALIA": "AU", "CANADA": "CA",
}


@dataclass(frozen=True)
class PiiConfig:
    geographies: tuple[str, ...] = ("IN",)
    language: str = "en"
    include_person: bool = True
    include_location: bool = False
    threshold: float = 0.35
    entity_thresholds: Mapping[str, float] = field(default_factory=dict)
    default_action: str = "redact"
    entity_actions: Mapping[str, str] = field(default_factory=dict)
    max_chars: int = 10_000

    def __post_init__(self):
        values = self.geographies
        if isinstance(values, str):
            values = (values,)
        if not isinstance(values, (tuple, list)) or not values:
            raise ValueError("Select at least one supported geography.")
        normalized = []
        for value in values:
            if not isinstance(value, str):
                raise ValueError("Geography must be a country code or name.")
            code = value.strip().upper()
            code = ALIASES.get(code, code)
            if code not in GEO_ENTITIES:
                raise ValueError("Unsupported geography; choose IN, US, GB, SG, AU or CA.")
            normalized.append(code)
        object.__setattr__(self, "geographies", tuple(sorted(set(normalized))))
        if self.language != "en":
            raise ValueError("This notebook supports English text only.")
        if type(self.include_person) is not bool or type(self.include_location) is not bool:
            raise ValueError("Entity switches must be booleans.")
        if type(self.max_chars) is not int or not 1 <= self.max_chars <= 100_000:
            raise ValueError("max_chars must be an integer from 1 to 100000.")
        if not isinstance(self.entity_thresholds, Mapping):
            raise ValueError("entity_thresholds must be a mapping.")
        if not isinstance(self.entity_actions, Mapping):
            raise ValueError("entity_actions must be a mapping.")
        thresholds = dict(self.entity_thresholds)
        actions = dict(self.entity_actions)
        if any(key not in self.entities for key in thresholds | actions):
            raise ValueError("An override refers to an inactive or unknown entity.")
        for score in (self.threshold, *thresholds.values()):
            if type(score) not in (int, float) or not isfinite(score) or not 0 <= score <= 1:
                raise ValueError("Thresholds must be finite numbers between 0 and 1.")
        if any(action not in ("redact", "block") for action in (self.default_action, *actions.values())):
            raise ValueError("Actions must be redact or block.")
        object.__setattr__(self, "entity_thresholds", MappingProxyType(thresholds))
        object.__setattr__(self, "entity_actions", MappingProxyType(actions))

    @property
    def entities(self) -> tuple[str, ...]:
        selected = set(COMMON_ENTITIES)
        for geography in self.geographies:
            selected.update(GEO_ENTITIES[geography])
        if self.include_person:
            selected.add("PERSON")
        if self.include_location:
            selected.add("LOCATION")
        return tuple(sorted(selected))

    def document(self) -> dict:
        return {
            "profile_version": PROFILE_VERSION,
            "geographies": list(self.geographies), "language": self.language,
            "include_person": self.include_person, "include_location": self.include_location,
            "entities": list(self.entities), "threshold": self.threshold,
            "entity_thresholds": dict(self.entity_thresholds),
            "default_action": self.default_action, "entity_actions": dict(self.entity_actions),
            "max_chars": self.max_chars,
        }

    @property
    def policy_id(self) -> str:
        payload = json.dumps(self.document(), sort_keys=True, separators=(",", ":"))
        return hashlib.sha256(payload.encode()).hexdigest()[:16]


@dataclass(frozen=True)
class Finding:
    entity: str
    start: int
    end: int
    score: float


@dataclass(frozen=True, repr=False)
class ScanResult:
    decision: str
    status: str
    release_text: str | None = field(repr=False)
    findings: tuple[Finding, ...]
    policy_id: str
    geographies: tuple[str, ...]
    latency_ms: float
    reason: str

    def audit(self) -> dict:
        return {
            "decision": self.decision, "status": self.status,
            "entity_counts": dict(Counter(item.entity for item in self.findings)),
            "policy_id": self.policy_id, "geographies": list(self.geographies),
            "latency_ms": round(self.latency_ms, 3), "reason": self.reason,
        }

    def __repr__(self) -> str:
        return f"ScanResult({self.audit()!r})"

In [ ]:
class OfflineEmailRecognizer(recognizers.EmailRecognizer):
    def __init__(self):
        super().__init__()
        self.suffixes = tldextract.TLDExtract(suffix_list_urls=(), cache_dir=None)

    def validate_result(self, pattern_text: str):
        return bool(self.suffixes(pattern_text).fqdn)


@lru_cache(maxsize=1)
def load_nlp_engine():
    if not spacy.util.is_package(MODEL_NAME):
        raise RuntimeError("Install en_core_web_sm before creating the guard.")
    provider = NlpEngineProvider(nlp_configuration={
        "nlp_engine_name": "spacy",
        "models": [{"lang_code": "en", "model_name": MODEL_NAME}],
    })
    return provider.create_engine()


def build_analyzer(config: PiiConfig) -> AnalyzerEngine:
    nlp = load_nlp_engine()
    registry = RecognizerRegistry(supported_languages=["en"])
    classes = {
        "IN": (recognizers.InPanRecognizer, recognizers.InAadhaarRecognizer,
               recognizers.InPassportRecognizer),
        "US": (recognizers.UsSsnRecognizer, recognizers.UsItinRecognizer,
               recognizers.UsBankRecognizer),
        "GB": (recognizers.UkNinoRecognizer, recognizers.NhsRecognizer),
        "SG": (recognizers.SgFinRecognizer,),
        "AU": (recognizers.AuTfnRecognizer, recognizers.AuMedicareRecognizer),
        "CA": (recognizers.CaSinRecognizer,),
    }
    selected = [OfflineEmailRecognizer, recognizers.CreditCardRecognizer,
                recognizers.IbanRecognizer, recognizers.IpRecognizer,
                recognizers.SpacyRecognizer]
    for geography in config.geographies:
        selected.extend(classes[geography])
    for recognizer_class in selected:
        registry.add_recognizer(recognizer_class())
    registry.add_recognizer(
        recognizers.PhoneRecognizer(supported_regions=list(config.geographies)))
    analyzer = AnalyzerEngine(registry=registry, nlp_engine=nlp, supported_languages=["en"])
    missing = set(config.entities) - set(analyzer.get_supported_entities(language="en"))
    if missing:
        raise RuntimeError("Installed recognizers do not cover the selected profile.")
    return analyzer


def merge_findings(findings: tuple[Finding, ...]) -> tuple[Finding, ...]:
    merged = []
    for item in sorted(findings, key=lambda value: (value.start, value.end)):
        if merged and item.start < merged[-1].end:
            previous = merged[-1]
            entity = previous.entity if previous.entity == item.entity else "PII"
            merged[-1] = Finding(entity, previous.start, max(previous.end, item.end),
                                 max(previous.score, item.score))
        else:
            merged.append(item)
    return tuple(merged)

In [ ]:
class GeographyPiiGuard:
    def __init__(self, config: PiiConfig, analyzer=None):
        self.config = config
        self.analyzer = analyzer if analyzer is not None else build_analyzer(config)
        self.anonymizer = AnonymizerEngine()

    def scan(self, text: str, *, language: str = "en") -> ScanResult:
        started = perf_counter()

        def outcome(decision, status, release_text, findings=(), reason=""):
            return ScanResult(decision, status, release_text, tuple(findings),
                              self.config.policy_id, self.config.geographies,
                              (perf_counter() - started) * 1000, reason)

        if not isinstance(text, str):
            return outcome("BLOCK", "ERROR", None, reason="INVALID_INPUT_TYPE")
        if language != self.config.language:
            return outcome("BLOCK", "UNSUPPORTED_INPUT", None, reason="UNSUPPORTED_LANGUAGE")
        if len(text) > self.config.max_chars:
            return outcome("BLOCK", "UNSUPPORTED_INPUT", None, reason="INPUT_TOO_LONG")
        if not text.strip():
            return outcome("ALLOW", "OK", text, reason="EMPTY_TEXT")
        try:
            detected = self.analyzer.analyze(
                text=text, language=language, entities=list(self.config.entities),
                score_threshold=0.0, return_decision_process=False,
            )
            findings = []
            for item in detected:
                if (item.entity_type not in self.config.entities
                        or type(item.start) is not int or type(item.end) is not int
                        or not 0 <= item.start < item.end <= len(text)
                        or not isfinite(item.score) or not 0 <= item.score <= 1):
                    raise ValueError("Invalid detector result.")
                threshold = self.config.entity_thresholds.get(item.entity_type, self.config.threshold)
                if item.score > 0 and item.score >= threshold:
                    findings.append(Finding(item.entity_type, item.start, item.end, float(item.score)))
            findings = tuple(sorted(set(findings), key=lambda item: (item.start, item.end, item.entity)))
            if not findings:
                return outcome("ALLOW", "OK", text, reason="NO_CONFIGURED_MATCH")
            if any(self.config.entity_actions.get(item.entity, self.config.default_action) == "block"
                   for item in findings):
                return outcome("BLOCK", "OK", None, findings, "BLOCK_POLICY")
            spans = merge_findings(findings)
            operators = {item.entity: OperatorConfig("replace", {"new_value": f"<{item.entity}>"})
                         for item in spans}
            redacted = self.anonymizer.anonymize(
                text=text,
                analyzer_results=[RecognizerResult(item.entity, item.start, item.end, item.score)
                                  for item in spans],
                operators=operators,
            ).text
            return outcome("REDACT", "OK", redacted, findings, "MATCHES_REPLACED")
        except Exception:
            # Detector exceptions may contain the original input.
            return outcome("BLOCK", "ERROR", None, reason="SCAN_FAILED")

    def manifest(self) -> dict:
        return {
            **self.config.document(), "policy_id": self.config.policy_id,
            "analyzer_version": version("presidio-analyzer"),
            "anonymizer_version": version("presidio-anonymizer"),
            "spacy_version": version("spacy"), "model": MODEL_NAME,
            "model_version": version(MODEL_NAME),
        }


def guarded_call(guard: GeographyPiiGuard, text: str, consumer):
    result = guard.scan(text)
    if result.status != "OK" or result.decision == "BLOCK":
        return result, None
    return result, consumer(result.release_text)

## 4. Configure your application
Edit this cell. `GEOGRAPHIES` accepts a country name/code or a list. For example, use `["IN", "US"]` when support records can contain both Indian and US identifiers.

Use `ENTITY_ACTIONS = {"IN_AADHAAR": "block"}` to block instead of replacing that entity. Overrides must refer to enabled entities. Higher thresholds reduce matches and can miss more PII; evaluate each change against labeled examples.

In [ ]:
GEOGRAPHIES = ["IN"]
ENTITY_THRESHOLDS = {}
ENTITY_ACTIONS = {}

config = PiiConfig(
    geographies=GEOGRAPHIES,
    include_person=True,
    include_location=False,
    threshold=0.35,
    entity_thresholds=ENTITY_THRESHOLDS,
    default_action="redact",
    entity_actions=ENTITY_ACTIONS,
    max_chars=10_000,
)
guard = GeographyPiiGuard(config)
print(json.dumps(guard.manifest(), indent=2))

The manifest records the active entities, thresholds, actions, and installed model/package versions. `policy_id` identifies configuration only; keep the manifest and your code revision together when releasing a policy. Top-level packages are pinned here; capture a full dependency lockfile for a production build.

In [ ]:
sample_text = "Refund my $10 order. Email alex@example.com. My PAN is ABCPD1234F."
result = guard.scan(sample_text)
print(result.audit())
print("Synthetic release:", result.release_text)

With the default India configuration, the email and PAN should be replaced. If you change geography, the India-specific check may no longer run. General recognizers can still match overlapping formats; exclusion from a country profile is not a promise that a value will pass unchanged.

### Country-by-country smoke samples
These fixtures use the default policy for each country independently of your edited configuration. They exercise every selected country-specific entity, the common entities, and name detection. `<PII>` on the ITIN example reflects overlapping ITIN/SSN matches.

In [ ]:
SAMPLES = [
    ("IN", "My PAN is ABCPD1234F.", "IN_PAN", "ABCPD1234F"),
    ("IN", "Aadhaar number: 234567890124.", "IN_AADHAAR", "234567890124"),
    ("IN", "Passport number: K1234567.", "IN_PASSPORT", "K1234567"),
    ("US", "My SSN is 321-54-6789.", "US_SSN", "321-54-6789"),
    ("US", "My ITIN is 912-70-1234.", "US_ITIN", "912-70-1234"),
    ("US", "Bank account: 123456789012.", "US_BANK_NUMBER", "123456789012"),
    ("GB", "National insurance number: AB123456C.", "UK_NINO", "AB123456C"),
    ("GB", "My NHS number is 943 476 5919.", "UK_NHS", "943 476 5919"),
    ("SG", "My NRIC is S1234567D.", "SG_NRIC_FIN", "S1234567D"),
    ("AU", "My TFN is 123456782.", "AU_TFN", "123456782"),
    ("AU", "Medicare number: 2123456701.", "AU_MEDICARE", "2123456701"),
    ("CA", "My SIN is 123 456 782.", "CA_SIN", "123 456 782"),
    ("GB", "Email alex@example.com.", "EMAIL_ADDRESS", "alex@example.com"),
    ("US", "Call +1 202-555-0142.", "PHONE_NUMBER", "+1 202-555-0142"),
    ("GB", "Card: 4111111111111111.", "CREDIT_CARD", "4111111111111111"),
    ("GB", "IBAN: GB82 WEST 1234 5698 7654 32.", "IBAN_CODE", "GB82 WEST 1234 5698 7654 32"),
    ("GB", "IP: 192.0.2.1.", "IP_ADDRESS", "192.0.2.1"),
    ("GB", "Contact John Smith about the missing invoice.", "PERSON", "John Smith"),
]
SAMPLE_GUARDS = {geo: GeographyPiiGuard(PiiConfig(geo)) for geo in GEO_ENTITIES}
for geography, sample, expected_entity, _ in SAMPLES:
    result = SAMPLE_GUARDS[geography].scan(sample)
    print(geography, expected_entity, result.decision, result.release_text)

## 5. Verify behavior, including failures
The tests below verify exact labeled spans and confirm that their values do not reach the released text. They also exercise profile unions, negative samples, configuration errors, input limits, entity switches, blocking, overlap handling, thresholds, scanner exceptions, and the downstream release gate.

Expected result: **15 tests pass, with 18 positive fixtures checked inside the first test**. This is a regression smoke suite, not a precision/recall benchmark. A passing suite does not establish coverage for your production distribution.

In [ ]:
import socket
import unittest
from unittest.mock import patch


class StubAnalyzer:
    def __init__(self, results=(), error=None):
        self.results = results
        self.error = error

    def analyze(self, **kwargs):
        if self.error:
            raise self.error
        return self.results


class PiiGuardTests(unittest.TestCase):
    def test_country_and_common_samples(self):
        for geography, text, entity, value in SAMPLES:
            with self.subTest(geography=geography, entity=entity):
                result = SAMPLE_GUARDS[geography].scan(text)
                self.assertEqual(result.status, "OK")
                self.assertEqual(result.decision, "REDACT")
                self.assertNotIn(value, result.release_text)
                start = text.index(value)
                self.assertTrue(any(f.entity == entity and f.start == start
                                    and f.end == start + len(value) for f in result.findings))

    def test_negative_samples(self):
        for text in ("What is your refund policy?", "Refund my $10 order.",
                     "Order reference: ORD-1234.", "Contact support tomorrow."):
            with self.subTest(text=text):
                result = SAMPLE_GUARDS["GB"].scan(text)
                self.assertEqual((result.status, result.decision), ("OK", "ALLOW"))
                self.assertEqual(result.release_text, text)

    def test_country_union(self):
        config = PiiConfig(["India", "USA", "IN"], include_person=False)
        self.assertEqual(config.geographies, ("IN", "US"))
        result = GeographyPiiGuard(config).scan("PAN: ABCPD1234F. SSN: 321-54-6789.")
        self.assertTrue({"IN_PAN", "US_SSN"} <= {f.entity for f in result.findings})
        self.assertNotIn("US_SSN", PiiConfig("IN").entities)

    def test_invalid_configuration(self):
        cases = ({"geographies": []}, {"geographies": "EU"}, {"language": "hi"},
                 {"threshold": float("nan")}, {"threshold": True}, {"max_chars": 0},
                 {"entity_actions": {"EMAIL_ADDRESS": "allow"}},
                 {"entity_thresholds": {"US_SSN": 0.5}})
        for values in cases:
            with self.subTest(values=values), self.assertRaises(ValueError):
                PiiConfig(**values)

    def test_invalid_input_is_not_released(self):
        guard = GeographyPiiGuard(PiiConfig("GB", max_chars=10))
        for text in (None, b"email", "x" * 11):
            result = guard.scan(text)
            self.assertEqual(result.decision, "BLOCK")
            self.assertIsNone(result.release_text)
        self.assertEqual(guard.scan("", language="hi").decision, "BLOCK")
        self.assertEqual(guard.scan("  ").release_text, "  ")

    def test_person_and_location_switches(self):
        config = PiiConfig("GB", include_person=False, include_location=True)
        guard = GeographyPiiGuard(config)
        self.assertNotIn("PERSON", config.entities)
        self.assertEqual(guard.scan("Contact John Smith.").decision, "ALLOW")
        self.assertTrue(any(f.entity == "LOCATION" for f in guard.scan("I live in London.").findings))

    def test_block_prevents_consumer_call(self):
        calls = []
        config = PiiConfig("US", entity_actions={"US_SSN": "block"})
        result, response = guarded_call(GeographyPiiGuard(config), SAMPLES[3][1], calls.append)
        self.assertEqual(result.decision, "BLOCK")
        self.assertIsNone(response)
        self.assertEqual(calls, [])

    def test_consumer_receives_only_redacted_text(self):
        calls = []
        result, _ = guarded_call(SAMPLE_GUARDS["GB"], "Email alex@example.com.", calls.append)
        self.assertEqual(calls, [result.release_text])
        self.assertNotIn("alex@example.com", calls[0])

    def test_failure_is_closed_and_diagnostics_exclude_payload(self):
        secret = "private@example.com"
        guard = GeographyPiiGuard(PiiConfig(), StubAnalyzer(error=RuntimeError(secret)))
        calls = []
        result, _ = guarded_call(guard, secret, calls.append)
        self.assertEqual((result.status, result.decision), ("ERROR", "BLOCK"))
        self.assertIsNone(result.release_text)
        self.assertEqual(calls, [])
        self.assertNotIn(secret, json.dumps(result.audit()) + repr(result))
        normal = SAMPLE_GUARDS["GB"].scan(secret)
        self.assertNotIn(secret, json.dumps(normal.audit()) + repr(normal))

    def test_malformed_detector_result_is_closed(self):
        malformed = RecognizerResult("EMAIL_ADDRESS", -1, 99, 0.9)
        guard = GeographyPiiGuard(PiiConfig(), StubAnalyzer([malformed]))
        self.assertEqual(guard.scan("abc").status, "ERROR")

    def test_overlapping_spans_are_fully_removed(self):
        detected = [RecognizerResult("PERSON", 0, 5, 0.8),
                    RecognizerResult("EMAIL_ADDRESS", 3, 8, 0.9)]
        guard = GeographyPiiGuard(PiiConfig(), StubAnalyzer(detected))
        self.assertEqual(guard.scan("abcdefgh safe").release_text, "<PII> safe")

    def test_threshold_override(self):
        detector = StubAnalyzer([RecognizerResult("EMAIL_ADDRESS", 0, 3, 0.4)])
        config = PiiConfig(entity_thresholds={"EMAIL_ADDRESS": 0.5})
        self.assertEqual(GeographyPiiGuard(config, detector).scan("abc").decision, "ALLOW")
        lower = replace(config, entity_thresholds={"EMAIL_ADDRESS": 0.3})
        self.assertEqual(GeographyPiiGuard(lower, detector).scan("abc").decision, "REDACT")

    def test_invalidated_zero_score_is_not_a_match(self):
        detector = StubAnalyzer([RecognizerResult("EMAIL_ADDRESS", 0, 3, 0.0)])
        guard = GeographyPiiGuard(PiiConfig(threshold=0), detector)
        self.assertEqual(guard.scan("abc").decision, "ALLOW")

    def test_local_scan_needs_no_network(self):
        with patch.object(socket.socket, "connect", side_effect=AssertionError("Network attempted")):
            guard = GeographyPiiGuard(PiiConfig("GB"))
            result = guard.scan("Email alex@example.com.")
            self.assertEqual((result.status, result.decision), ("OK", "REDACT"))

    def test_policy_id_changes_with_configuration(self):
        self.assertEqual(PiiConfig("UK").policy_id, PiiConfig("GB").policy_id)
        self.assertNotEqual(PiiConfig("GB").policy_id, PiiConfig("GB", threshold=0.6).policy_id)


suite = unittest.defaultTestLoader.loadTestsFromTestCase(PiiGuardTests)
verification = unittest.TextTestRunner(verbosity=1).run(suite)
assert verification.wasSuccessful(), "Verification failed; do not use this configuration."
print(f"Verified {verification.testsRun} tests; {len(SAMPLES)} labeled positive samples.")

### Add your own labeled sample
Use fabricated or access-controlled evaluation data. Add cases with missing context, OCR errors, Unicode substitutions, broken formatting, uncommon names, and realistic non-PII lookalikes. Keep a separate held-out set when tuning thresholds.

For a real evaluation, annotate every relevant span and measure recall per entity/geography, false positives on ordinary requests, and residual sensitive spans in the released text. A checksum validates a format, not ownership or whether a malformed identifier is still sensitive. The default detectors can miss obfuscated, invalid-checksum, or unfamiliar PII.

In [ ]:
CUSTOM_GEOGRAPHIES = ["IN", "US"]
CUSTOM_TEXT = "Email alex@example.com. PAN: ABCPD1234F. SSN: 321-54-6789."
EXPECTED_VALUES = ["alex@example.com", "ABCPD1234F", "321-54-6789"]

custom_guard = GeographyPiiGuard(PiiConfig(CUSTOM_GEOGRAPHIES))
custom_result = custom_guard.scan(CUSTOM_TEXT)
assert custom_result.status == "OK"
assert custom_result.decision == "REDACT"
assert all(value not in custom_result.release_text for value in EXPECTED_VALUES)
print(custom_result.audit())
print("Custom synthetic sample passed.")

## 6. Measure latency on your machine
This small benchmark measures warmed, sequential scans. Model loading is excluded. It reports input size and median/p95 scan time; it does not establish a service latency SLO. Repeat with representative lengths, concurrency, hardware, and country combinations before sizing a deployment.

In [ ]:
from statistics import median
from math import ceil

benchmark_text = "Refund my $10 order. Email alex@example.com. PAN: ABCPD1234F."
benchmark_guard = GeographyPiiGuard(PiiConfig("IN"))
benchmark_guard.scan(benchmark_text)
timings = []
for _ in range(30):
    measured = benchmark_guard.scan(benchmark_text)
    assert measured.status == "OK" and measured.decision == "REDACT"
    timings.append(measured.latency_ms)
print({
    "iterations": len(timings),
    "characters": len(benchmark_text),
    "median_ms": round(median(timings), 2),
    "p95_ms": round(sorted(timings)[ceil(len(timings) * 0.95) - 1], 2),
})

## 7. Put the gate before the boundary
Use the released text for model requests, retrieved excerpts, or tool payloads that must exclude the configured identifiers. Keep the original outside traces, exception messages, and callback logs. A second scan may be needed before releasing the generated answer because retrieved data or generation can introduce new PII.

The mock consumer below represents an LLM call. It receives only the released text. A blocked or failed scan never invokes it. Replacing text does not authorize a refund or any other tool action; resource ownership and transaction permissions remain separate controls.

In [ ]:
received = []

def mock_llm(text: str) -> str:
    received.append(text)
    return "Please provide the order reference to continue."

scan, answer = guarded_call(guard, "Refund my $10 order. Email alex@example.com.", mock_llm)
if scan.status != "OK" or scan.decision == "BLOCK":
    assert not received and answer is None
else:
    assert received == [scan.release_text]
    assert "alex@example.com" not in received[0]
print({"decision": scan.decision, "consumer_calls": len(received), "answer": answer})

### Production integration decisions
- **Language and geography:** Select a trusted baseline profile; add source countries when needed. Reject or route unsupported languages instead of silently using English NER. Add and test each recognizer before adding a geography to the profile map.
- **Timing:** This is a synchronous release gate. A FastAPI `async` handler should offload CPU work to bounded workers; `async def` alone does not make scanning nonblocking. Do not run mandatory PII checks in the background after releasing data.
- **Capacity and failure:** Initialize models once per worker, bound input size and queues, and set service deadlines from measured latency. This notebook has no hard wall-clock cancellation. An async timeout does not necessarily stop CPU work; use worker/process supervision where cancellation matters. On overload or timeout, release no payload or route to an approved local fallback.
- **Long documents and streaming:** Do not truncate and forward the unchecked remainder. Chunking needs overlap and span reconciliation; a sensitive value can straddle chunks. Buffer output until the required check passes. This notebook scans complete strings only.
- **Privacy:** Keep raw and redacted payloads out of telemetry by default. The supplied audit method is deliberately limited. Review framework tracing, access logs, notebook outputs, and error capture separately; the result wrapper cannot control them. Clear outputs before using real data.
- **Evaluation:** Include false positives, malformed identifiers, mixed-country records, and missed names in release tests. Treat this as a configurable baseline, not an anonymization guarantee. Removing detected PII does not prevent re-identification from remaining context.

### References
[Presidio supported entities](https://presidio.dataprivacystack.org/supported_entities/) · [Country filtering](https://presidio.dataprivacystack.org/analyzer/filtering_by_country/) · [Language support](https://presidio.dataprivacystack.org/analyzer/languages/) · [Presidio source](https://github.com/microsoft/presidio)